# Data and models
Goal: estimate the skill gap between two teams. Every model produces a **rating gap** (home strength minus away strength, home advantage included). One shared mapping turns that gap into **win/draw/loss probabilities** (ordered logit) and an **expected goal difference** (a straight line), so all models are scored the same way.

A model is a plain Python dict plus three functions (`gap`, `update`, `start_season`) in `ratings/models/`. `base.walk_forward` goes through matches in date order, records the gap *before* each match, then lets the model learn the result, so no model sees a result it is predicting.

| Family | Models | Idea |
|---|---|---|
| Elo | `elo_none`, `elo_sqrt` (chosen), `elo_wfe_autocorr`, `elo_xg` | logistic expectation; margin-of-victory scaling; xG-blended result |
| Pi-ratings | `pi` | separate home/away ratings updated on goal-difference error |
| Glicko | `glicko` | rating plus uncertainty, growing with idle time and at season boundaries |
| Dixon-Coles | `dixon_coles`, `dixon_coles_xg` | time-decayed Poisson attack/defence ratings |
| Ridge | `ridge_gd`, `ridge_gd_xg` | batch weighted ridge regression on goal difference |
| Ensemble | `ensemble_ols` | OLS stack of all gaps (weights fit on training seasons only) |

In [1]:
%matplotlib inline
import sys, json, pathlib, warnings
warnings.filterwarnings("ignore")
# make the repo root importable no matter which folder the notebook is opened from
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "ratings").is_dir())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from ratings import base
from ratings.data import load_matches
from ratings.tune import PARAMS_PATH, load_best
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Matches
Regular-season matches with a score, men and women rated separately. `hxg`/`axg` are team xG (missing for some matches).

In [2]:
matches = {g: load_matches(g) for g in ("male", "female")}
for g, df in matches.items():
    print(g, len(df), "matches; xG coverage", f"{df['hxg'].notna().mean():.0%}")
    display(df.groupby("year").agg(matches=("match_id", "size"), teams=("home", "nunique"),
                                   home_win_rate=("hg", lambda s: (s > df.loc[s.index, "ag"]).mean()),
                                   goals_per_match=("hg", lambda s: (s + df.loc[s.index, "ag"]).mean())).round(3))

male 13358 matches; xG coverage 87%


,matches,teams,home_win_rate,goals_per_match
year,,,,
2017,47,36,0.213,2.404
2019,1025,206,0.532,2.869
2020,114,35,0.474,2.798
2021,2780,214,0.483,2.814
2022,1944,214,0.477,2.838
2023,1938,214,0.485,2.869
2024,1971,215,0.482,2.901
2025,1960,216,0.481,2.790
2026,1579,213,0.497,3.049


female 21568 matches; xG coverage 85%


,matches,teams,home_win_rate,goals_per_match
year,,,,
2019,675,128,0.567,2.644
2020,326,77,0.485,2.626
2021,4483,353,0.496,2.681
2022,3136,350,0.463,2.663
2023,3348,359,0.472,2.778
2024,3380,352,0.496,2.753
2025,3440,361,0.462,2.738
2026,2780,354,0.473,2.824


## 2. The models and their settings
Each model's name, kind, fixed settings and the values tried when tuning (`ratings/specs.py`).

In [3]:
from ratings.specs import SPECS, BATCH
display(pd.DataFrame([{"model": n, "kind": k, "fixed settings": str(f), "tuned over": ", ".join(s)} for n, (k, f, s) in SPECS.items()]).set_index("model"))

,kind,fixed settings,tuned over
model,,,
elo_none,elo,{'mov': 'none'},"k, hfa, carry"
elo_sqrt,elo,{'mov': 'sqrt'},"k, hfa, carry"
elo_wfe_autocorr,elo,"{'mov': 'wfe', 'autocorr': True}","k, hfa, carry"
elo_xg,elo,{},"k, hfa, carry, xg_weight, xg_c"
pi,pi,{},"lam, gamma, c, carry"
glicko,glicko,{},"hfa, c_week, season_c, carry, rd_max"
dixon_coles,dixon_coles,{},"half_life_days, ridge"
dixon_coles_xg,dixon_coles,{'target': 'xg'},"half_life_days, ridge, xg_weight"
ridge_gd,ridge_gd,{},"half_life_days, ridge"


## 3. A model is just a dict
Build one, feed it matches, and look at its state.

In [4]:
from ratings.specs import build
from ratings.models import elo
model = build("elo_sqrt", load_best("male")["elo_sqrt"]["params"])
predictions = base.walk_forward(model, matches["male"])
print({k: v for k, v in model.items() if k != "ratings"})
top = pd.Series(model["ratings"]).sort_values(ascending=False).head(5)
print("highest Elo ratings (team ids):", top.round(0).to_dict())
display(predictions.tail(5))

{'kind': 'elo', 'name': 'elo_sqrt', 'k': 60, 'hfa': 70, 'mov': 'sqrt', 'xg_weight': 0.0, 'xg_c': 1.2, 'carry': 0.9, 'autocorr': False}
highest Elo ratings (team ids): {27948: 2054.0, 28651: 2017.0, 27990: 1912.0, 28672: 1910.0, 26951: 1887.0}


,match_id,year,date,home,away,hg,ag,hxg,axg,gap
13353,5883137,2026,2026-10-27 23:00:00,28358,28319,2,2,1.13,0.63,31.876758
13354,5887141,2026,2026-10-27 23:00:00,24698,28348,1,3,0.87,1.24,-383.728356
13355,5880147,2026,2026-10-28 02:00:00,24424,66399,1,4,2.08,1.65,2.353993
13356,5865539,2026,2026-10-28 15:00:00,28438,28432,2,1,2.66,1.69,277.709517
13357,5865540,2026,2026-10-28 18:00:00,66850,28878,0,0,NaN,NaN,-52.655704


## 4. Checks on made-up matches
`test_ratings.py` plays a fake league where every team's true strength is known and checks that each model recovers it, that the first gap is only the home advantage (no look-ahead) and that the probabilities behave.

In [5]:
import subprocess
out = subprocess.run([sys.executable, "-m", "pytest", "ratings/test_ratings.py", "-q"], cwd=ROOT, capture_output=True, text=True)
print(out.stdout[-600:])

...                                                                      [100%]
3 passed in 0.96s

